# Gen-axis SR comparison figures

Per-**generation** comparison plots for the symbolic-regression evolution runs.
All the plotting machinery lives in `gen_axis_plots.py`; this notebook only holds
the per-figure *commands* (which run ids, which panels, which title) so each
figure can be browsed / tweaked / re-run on its own.

- `gp.load_method(api, widx, [ids...], tag)` -> per-seed entries for one method
- `gp.render(methods, out_path, suptitle, panels=...)` -> the panel grid
  - `methods` = list of `(name, runs, color, marker)`
  - `panels`  = `gp.PANELS_COMPARISON` (multi-method, bar decomp) or
    `gp.PANELS_SINGLE` (one run: best-current + re-evals + table decomp)

Re-run the setup cell after editing `gen_axis_plots.py` (autoreload picks it up).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import matplotlib.pyplot as plt
import gen_axis_plots as gp

api = gp.get_api()
wandb_index = gp.build_wandb_index()

: 

## Run ids

Edit these lists to swap seeds in/out. Each is one method (a set of seeds).
`reeval=none`, `offspring=20` throughout unless noted.

In [ ]:
N1_NONE  = [89281, 825769, 825773, 825777, 825781]   # nn_n1o20   seeds 0-4
N3_NONE  = [89282, 825770, 825774, 825778, 825782]   # nn_n3o20   seeds 0-4
N10_NONE = [568245, 568246]                          # nn_n10o20  seeds 0-1 (still running)
# smart-reeval n1: offspring 5 + smart reeval, B=20 evals/gen (budget-matched to n1o20)
SMART_N1 = [825767, 825771, 825775, 825779, 825783]  # sm_n1o5b20 seeds 0-4
BEST_RUN = 538190

## n1 vs n3 vs n10 (all reeval=none, offspring=20)

Same #offspring/gen for all; per generation isolates the effect of averaging each
offspring over more seeds (n3 pays 3x, n10 pays 10x eval cost -- factored out here).

In [ ]:
n1  = gp.load_method(api, wandb_index, N1_NONE,  "n1")
n3  = gp.load_method(api, wandb_index, N3_NONE,  "n3")
n10 = gp.load_method(api, wandb_index, N10_NONE, "n10")

gp.render(
    [("n1 (n_runs 1)",  n1,  gp.COLOR(0), "o"),
     ("n3 (n_runs 3)",  n3,  gp.COLOR(3), "s"),
     ("n10 (n_runs 10)", n10, gp.COLOR(2), "^")],
    gp.OUTDIR / "gen_axis_n1_vs_n3.png",
    "n1 vs n3 vs n10 (all reeval=none, offspring=20) — per generation\n"
    "same #offspring/gen for all; n3/n10 pay 3x/10x eval cost (factored out here)",
    panels=gp.PANELS_COMPARISON,
)
plt.show()

## smart-n1 vs n3

smart n1 spends the SAME 20 evals/gen as n1o20 but on 5 offspring + smart reeval;
n3 spends 60 evals/gen (20 offspring x 3 seeds). If smart n1 tracks n3 here, smart
reeval matches brute-force averaging far more cheaply.

In [ ]:
smart_n1 = gp.load_method(api, wandb_index, SMART_N1, "smart n1")

gp.render(
    [("smart n1 (o5, B=20)",      smart_n1, gp.COLOR(2), "^"),
     ("n3 (n_runs 3, o20, B=60)", n3,       gp.COLOR(3), "s")],
    gp.OUTDIR / "gen_axis_smart_n1_vs_n3.png",
    "smart-n1 vs n3 — per generation\n"
    "smart n1: offspring 5 + smart reeval, 20 evals/gen;  "
    "n3: offspring 20 x 3 seeds, 60 evals/gen",
    panels=gp.PANELS_COMPARISON,
)
plt.show()

## Single best run (538190)

One run, no comparison. Uses `PANELS_SINGLE`: "best score (current posteriors)"
(can drop when a re-eval lowers the top candidate), a re-evals-per-gen panel, and
the train-score decomposition as a table.

In [ ]:
best = gp.load_method(api, wandb_index, [BEST_RUN], "best")

gp.render(
    [(f"run {BEST_RUN}", best, gp.COLOR(0), "o")],
    gp.OUTDIR / "gen_axis_best_run.png",
    f"Best run ({BEST_RUN}) — per generation",
    panels=gp.PANELS_SINGLE,
)
plt.show()